In [1]:
#load the necessary libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score




In [2]:
#load the clinical and methylation train datasets
clinical_df_train = pd.read_csv('../data/processed/clinical_kirc_train_clustered_pca.csv')
methylation_df_train = pd.read_parquet('../data/processed/methylation_kirc_train.parquet')


In [3]:
#load the clinical and methylation holdout datasets
clinical_df_holdout = pd.read_csv('../data/processed/clinical_kirc_holdout.csv')
methylation_df_holdout = pd.read_parquet('../data/processed/methylation_kirc_holdout.parquet')

In [4]:
print("This is what the training clinical data looks like:")
clinical_df_train

This is what the training clinical data looks like:


,Unnamed: 0,sample,tissue_type.samples,kmeans_pca_2_k2,kmeans_pca_3_k2,kmeans_pca_opt_k2,kmeans_raw_k2,hierarchical_pca_2_k2,hierarchical_pca_3_k2,hierarchical_pca_opt_k2,hierarchical_raw_k2,gmm_pca_2_k2,gmm_pca_3_k2,gmm_pca_opt_k2
0,5,TCGA-DV-5573-01A,Tumor,0,0,0,1,0,0,0,0,0,1,0
1,8,TCGA-BP-4795-01A,Tumor,1,1,1,1,0,0,0,1,0,1,0
2,9,TCGA-BP-4795-11A,Normal,1,1,1,1,1,1,1,1,1,0,1
3,26,TCGA-DV-5565-01A,Tumor,0,0,0,1,0,0,0,0,0,1,0
4,27,TCGA-CJ-4869-11A,Normal,1,1,1,1,1,1,1,1,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
280,940,TCGA-A3-3385-11A,Normal,1,1,1,1,1,1,1,1,1,0,1
281,941,TCGA-A3-3385-01A,Tumor,0,0,0,1,0,0,0,0,0,1,0
282,942,TCGA-B8-5552-01B,Tumor,0,0,0,1,0,0,0,0,0,1,0
283,946,TCGA-B0-4821-01A,Tumor,0,0,0,1,0,0,0,0,0,1,0


In [5]:
print("This is what the training methylation data looks like:")
methylation_df_train

This is what the training methylation data looks like:


Composite Element REF,cg00000108,cg00000236,cg00000292,cg00000321,cg00000363,cg00000622,cg00000658,cg00000714,cg00000721,cg00000734,...,ctl_69667417,ctl_70664314,ctl_70700334,ctl_71670310,ctl_71678368,ctl_71718498,ctl_73635489,ctl_73784382,ctl_73794434,ctl_74666473
index,,,,,,,,,,,,,,,,,,,,,
TCGA-B0-5108-01A,0.972311,0.909915,0.601544,0.283628,0.221306,0.008769,0.857267,0.132064,0.942689,0.064539,...,0.921408,0.074574,0.067612,0.954321,0.057296,0.196588,0.061329,0.943881,0.950757,0.948703
TCGA-B0-5703-01A,0.964033,0.874732,0.590178,0.454810,0.209447,0.013945,0.931355,0.189218,0.943166,0.063036,...,0.815832,0.058441,0.144345,0.932150,0.056047,0.163352,0.077233,0.903676,0.916601,0.964794
TCGA-CJ-4905-11A,0.960496,0.907699,0.489555,0.437536,0.160331,0.007906,0.914621,0.193818,0.957598,0.061109,...,0.912401,0.043139,0.053916,0.964921,0.091277,0.189204,0.080636,0.940926,0.950549,0.958856
TCGA-B0-4714-01A,0.959082,0.892050,0.539843,0.709672,0.251050,0.010895,0.853589,0.116101,0.945019,0.069490,...,0.914239,0.048749,0.074723,0.960944,0.065973,0.224263,0.061164,0.936791,0.928389,0.939751
TCGA-B2-5635-01A,0.971277,0.933855,0.387262,0.299354,0.242985,0.014074,0.884547,0.176327,0.962483,0.081663,...,0.924102,0.026477,0.071569,0.965354,0.055488,0.140422,0.065160,0.939127,0.953792,0.961929
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TCGA-BP-4993-01A,0.965083,0.926505,0.766406,0.518425,0.185070,0.010854,0.831226,0.119747,0.958758,0.043677,...,0.940402,0.015461,0.051947,0.953817,0.047797,0.175196,0.049526,0.955170,0.952150,0.938029
TCGA-CZ-4865-01A,0.969403,0.915849,0.515882,0.515539,0.205803,0.009411,0.901285,0.141882,0.968379,0.055377,...,0.959454,0.018266,0.076597,0.968557,0.044430,0.154470,0.047228,0.961375,0.963338,0.960697
TCGA-BP-4782-11A,0.959676,0.936125,0.496571,0.396565,0.258784,0.009743,0.836323,0.200536,0.950403,0.067827,...,0.913990,0.028057,0.060892,0.967519,0.053769,0.205141,0.057893,0.949411,0.947987,0.945548


In [6]:
print("This is what the holdout clinical data looks like:")
clinical_df_holdout

This is what the holdout clinical data looks like:


,Unnamed: 0,sample,tissue_type.samples
0,0,TCGA-B0-5695-01A,Tumor
1,24,TCGA-B0-4712-11A,Normal
2,25,TCGA-B0-4712-01A,Tumor
3,43,TCGA-CZ-4856-01A,Tumor
4,44,TCGA-CZ-4856-11A,Normal
...,...,...,...
193,931,TCGA-B2-3924-01B,Tumor
194,944,TCGA-B0-5402-01A,Tumor
195,945,TCGA-B0-5402-11A,Normal
196,948,TCGA-CZ-4866-01A,Tumor


In [7]:
print("This is what the holdout methylation data looks like:")
methylation_df_holdout

This is what the holdout methylation data looks like:


Composite Element REF,cg00000108,cg00000236,cg00000292,cg00000321,cg00000363,cg00000622,cg00000658,cg00000714,cg00000721,cg00000734,...,ctl_69667417,ctl_70664314,ctl_70700334,ctl_71670310,ctl_71678368,ctl_71718498,ctl_73635489,ctl_73784382,ctl_73794434,ctl_74666473
index,,,,,,,,,,,,,,,,,,,,,
TCGA-B4-5834-01A,0.972071,0.917789,0.426973,0.270070,0.351346,0.011284,0.862835,0.141163,0.965383,0.074103,...,0.889554,0.100888,0.068487,0.951324,0.048633,0.249223,0.052203,0.951095,0.959078,0.961796
TCGA-DV-5575-01A,0.964997,0.931517,0.492524,0.310941,0.226339,0.012376,0.874163,0.160123,0.960429,0.067214,...,0.925697,0.037106,0.071529,0.960663,0.044961,0.147622,0.050874,0.937494,0.958510,0.949927
TCGA-BP-5195-11A,0.972698,0.888925,0.512414,0.574735,0.201092,0.024069,0.912388,0.196851,0.956501,0.080242,...,0.946776,0.027305,0.059446,0.964031,0.060363,0.179446,0.062773,0.946509,0.943428,0.962431
TCGA-BP-5195-01A,0.969650,0.927199,0.461449,0.447459,0.224090,0.008947,0.954651,0.139885,0.960948,0.077233,...,0.943960,0.018929,0.062069,0.967989,0.047160,0.172492,0.048383,0.957683,0.951249,0.963472
TCGA-B8-5165-01A,0.967205,0.916345,0.550989,0.284367,0.180671,0.008042,0.877757,0.124144,0.959560,0.048545,...,0.940954,0.029113,0.073797,0.968861,0.043066,0.169768,0.045810,0.957088,0.956074,0.949140
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TCGA-B0-4813-11A,0.956692,0.906532,0.493675,0.550481,0.114842,0.010328,0.848741,0.187480,0.935683,0.056698,...,0.894490,0.013092,0.086768,0.939911,0.109435,0.202068,0.096711,0.910936,0.899721,0.951262
TCGA-BP-5010-11A,0.967596,0.910573,0.564515,0.533479,0.155655,0.009449,0.844393,0.163511,0.955114,0.057939,...,0.944623,0.050170,0.055988,0.967544,0.049648,0.170393,0.053387,0.957767,0.960849,0.950621
TCGA-B0-4823-11A,0.967950,0.911225,0.522083,0.512706,0.230194,0.009638,0.822798,0.171498,0.950986,0.055662,...,0.930867,0.030663,0.066078,0.965939,0.050796,0.222552,0.061422,0.950035,0.951596,0.953344


In [8]:

#rename the index column to 'sample' if it exists in the methylation dataframe
if 'Composite Element REF' in methylation_df_train.columns:
    methylation_df_train.rename(columns={'index': 'sample'}, inplace=True)
elif methylation_df_train.index.name == 'index':
    methylation_df_train.index.name = 'sample'
    methylation_df_train = methylation_df_train.reset_index()

#do the same for the holdout methylation dataframe so it can be aligned with clinical_df_holdout by sample ID
if 'Composite Element REF' in methylation_df_holdout.columns:
    methylation_df_holdout.rename(columns={'index': 'sample'}, inplace=True)
elif methylation_df_holdout.index.name == 'index':
    methylation_df_holdout.index.name = 'sample'
    methylation_df_holdout = methylation_df_holdout.reset_index()



In [ ]:

#ClaudeCode Ver. 1: automatically use whichever clustering column scored highest ARI in
#02-1_clustering_pca.ipynb, instead of hardcoding a column name by hand
with open('../models/best_pseudo_label_column.txt') as f:
    best_pseudo_label_col = f.read().strip()
print(f"Using best pseudo-label column: {best_pseudo_label_col}")

#inner join ensures we only train on samples present in both datasets
data = pd.merge(
    clinical_df_train[['sample', best_pseudo_label_col, 'tissue_type.samples']],
    methylation_df_train,
    on='sample',
    how='inner'
)

print(f"Data merged successfully. Total samples to process: {len(data)}")

In [ ]:
#prepare Features (X) and Targets (y1, y2)
X = data.drop(columns=['sample', best_pseudo_label_col, 'tissue_type.samples'])

#target 1: pseudo-labels from clustering (Binary -> No encoding needed)
y1 = data[best_pseudo_label_col].astype(int)

#target 2: ground truth tissue_type.samples (Categorical -> Needs encoding)
le = LabelEncoder()
y2 = le.fit_transform(data['tissue_type.samples']) #change the 'data' to the holdout with labels attached

#cluster IDs (0/1) from unsupervised clustering are arbitrary -- ARI is invariant to which physical
#cluster gets called "0" vs "1", but training a probabilistic classifier and scoring predict_proba[:, 1]
#against the ground truth is NOT: if the pseudo-label's "1" happens to mean the opposite tissue type
#from the ground truth's "1", AUC comes out near 0 (a near-perfect INVERTED ranking) instead of near 1.
#align the pseudo-label direction to the ground truth via majority vote on the training data (the same
#data ARI was already computed on in 02-1_clustering_pca.ipynb, so this uses no new information)
#edited by claude code
if (y1 == y2).mean() < 0.5:
    print("Pseudo-label cluster direction is inverted relative to ground truth -- flipping 0/1 to align")
    y1 = 1 - y1


In [11]:

#initialize the models 

svm_truth = SVC(kernel='rbf', probability=True, random_state=67)
svm_pseudo = SVC(kernel='rbf', probability=True, random_state=67)


rf_truth = RandomForestClassifier(n_estimators=100, random_state=67)
rf_pseudo = RandomForestClassifier(n_estimators=100, random_state=67)



In [12]:
# --- Scenario B: Train on Ground Truth (y2), Test on Ground Truth (y2) ---
print("Train SVM with pseudo-labels and ground truth labels")
svm_pseudo.fit(X, y1)
svm_truth.fit(X, y2)

print("Train RF with pseudo-labels and ground truth labels")
rf_pseudo.fit(X, y1)
rf_truth.fit(X, y2)

print("Models trained successfully.")

Train SVM with pseudo-labels and ground truth labels
Train RF with pseudo-labels and ground truth labels
Models trained successfully.


In [13]:
#align the holdout methylation and clinical data by sample ID
#(methylation_df_holdout and clinical_df_holdout are NOT in the same row order, so a merge is required
#before evaluating predictions against ground truth labels)
print("Aligning holdout methylation and clinical data by sample ID...")
holdout_data = pd.merge(
    clinical_df_holdout[['sample', 'tissue_type.samples']],
    methylation_df_holdout,
    on='sample',
    how='inner'
)
print(f"Holdout data aligned successfully. Total samples to process: {len(holdout_data)}")

X_holdout = holdout_data.drop(columns=['sample', 'tissue_type.samples'])

Aligning holdout methylation and clinical data by sample ID...
Holdout data aligned successfully. Total samples to process: 198


In [14]:
#label encode the holdout labels using the SAME encoder fitted on the training data,
#now correctly row-aligned with X_holdout via the merge above
y2_holdout = le.transform(holdout_data['tissue_type.samples'])

In [15]:
print("Predict the holdout data using the trained models")

#predict probabilities using SVM trained on pseudo-labels and ground truth labels respectively
svm_probs_pseudo = svm_pseudo.predict_proba(X_holdout)[:, 1]
svm_probs_truth = svm_truth.predict_proba(X_holdout)[:, 1]

#predict probabilities using RF trained on pseudo-labels and ground truth labels respectively
rf_probs_pseudo = rf_pseudo.predict_proba(X_holdout)[:, 1]
rf_probs_truth = rf_truth.predict_proba(X_holdout)[:, 1]

Predict the holdout data using the trained models


In [16]:
#evaluate SVM models against GROUND TRUTH (y2_test)
svm_auc_pseudo = roc_auc_score(y2_holdout, svm_probs_pseudo)
svm_auc_truth = roc_auc_score(y2_holdout, svm_probs_truth)

#evaluate RF models against GROUND TRUTH (y2_test)
rf_auc_pseudo = roc_auc_score(y2_holdout, rf_probs_pseudo)
rf_auc_truth = roc_auc_score(y2_holdout, rf_probs_truth)

In [ ]:
#compare
print("\n=== Final AUC Comparison (Evaluated on Ground Truth Test Set) ===")
print(f"Trained on pseudo-labels: {best_pseudo_label_col} (Pseudo) | SVM AUC: {svm_auc_pseudo:.4f} | RF AUC: {rf_auc_pseudo:.4f} ")
print(f"Trained on ground truth: tissue_type.samples   | SVM AUC: {svm_auc_truth:.4f} | RF AUC: {rf_auc_truth:.4f} ")

In [18]:
#compare classifier performance using PCA-reduced features instead of the raw ~371k methylation probes
#we reuse the SAME PCA fit on the training data in 02-1_clustering_pca.ipynb (n=3) rather than fitting a
#new PCA on the holdout set -- PCA learns a coordinate system from whatever data it's fit on, so a PCA fit
#separately on holdout would express its components in different axes (not comparable to the train PCA),
#and would also leak holdout distribution information into preprocessing, defeating the point of a holdout
import joblib

pca_model = joblib.load('../models/pca_best_model.joblib')
print(f"Loaded PCA model with n_components={pca_model.n_components_}")

Loaded PCA model with n_components=3


In [19]:
#fit the scaler ONLY on the training methylation features (never on holdout), then transform both splits with it
pca_scaler = StandardScaler()
X_scaled = pca_scaler.fit_transform(X)
X_holdout_scaled = pca_scaler.transform(X_holdout)

#project both splits into the same PCA space using the PCA model fit on training data
X_pca = pca_model.transform(X_scaled)
X_holdout_pca = pca_model.transform(X_holdout_scaled)

print(f"Train PCA-reduced shape: {X_pca.shape}, Holdout PCA-reduced shape: {X_holdout_pca.shape}")

Train PCA-reduced shape: (285, 3), Holdout PCA-reduced shape: (198, 3)


/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: divide by zero encountered in matmul
  X_transformed = X @ self.components_.T
/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: overflow encountered in matmul
  X_transformed = X @ self.components_.T
/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: invalid value encountered in matmul
  X_transformed = X @ self.components_.T
/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: divide by zero encountered in matmul
  X_transformed = X @ self.components_.T
/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: overflow encountered in matmul
  X_transformed = X @ self.components_.T
/Users/matibai-tcm/Library/Python/3.9/lib/python

In [20]:
#train fresh SVM/RF classifiers on the PCA-reduced features (same targets y1/y2 as before)
svm_truth_pca = SVC(kernel='rbf', probability=True, random_state=67)
svm_pseudo_pca = SVC(kernel='rbf', probability=True, random_state=67)

rf_truth_pca = RandomForestClassifier(n_estimators=100, random_state=67)
rf_pseudo_pca = RandomForestClassifier(n_estimators=100, random_state=67)

print("Train SVM (PCA-reduced) with pseudo-labels and ground truth labels")
svm_pseudo_pca.fit(X_pca, y1)
svm_truth_pca.fit(X_pca, y2)

print("Train RF (PCA-reduced) with pseudo-labels and ground truth labels")
rf_pseudo_pca.fit(X_pca, y1)
rf_truth_pca.fit(X_pca, y2)

print("PCA-reduced models trained successfully.")

Train SVM (PCA-reduced) with pseudo-labels and ground truth labels
Train RF (PCA-reduced) with pseudo-labels and ground truth labels
PCA-reduced models trained successfully.


In [21]:
#predict on the PCA-reduced holdout data
svm_probs_pseudo_pca = svm_pseudo_pca.predict_proba(X_holdout_pca)[:, 1]
svm_probs_truth_pca = svm_truth_pca.predict_proba(X_holdout_pca)[:, 1]

rf_probs_pseudo_pca = rf_pseudo_pca.predict_proba(X_holdout_pca)[:, 1]
rf_probs_truth_pca = rf_truth_pca.predict_proba(X_holdout_pca)[:, 1]

svm_auc_pseudo_pca = roc_auc_score(y2_holdout, svm_probs_pseudo_pca)
svm_auc_truth_pca = roc_auc_score(y2_holdout, svm_probs_truth_pca)

rf_auc_pseudo_pca = roc_auc_score(y2_holdout, rf_probs_pseudo_pca)
rf_auc_truth_pca = roc_auc_score(y2_holdout, rf_probs_truth_pca)

print("\n=== AUC Comparison: Raw Features vs PCA-Reduced Features (Evaluated on Ground Truth Test Set) ===")
print(f"Raw features   | Pseudo-labels | SVM AUC: {svm_auc_pseudo:.4f} | RF AUC: {rf_auc_pseudo:.4f}")
print(f"Raw features   | Ground truth  | SVM AUC: {svm_auc_truth:.4f} | RF AUC: {rf_auc_truth:.4f}")
print(f"PCA-reduced    | Pseudo-labels | SVM AUC: {svm_auc_pseudo_pca:.4f} | RF AUC: {rf_auc_pseudo_pca:.4f}")
print(f"PCA-reduced    | Ground truth  | SVM AUC: {svm_auc_truth_pca:.4f} | RF AUC: {rf_auc_truth_pca:.4f}")


=== AUC Comparison: Raw Features vs PCA-Reduced Features (Evaluated on Ground Truth Test Set) ===
Raw features   | Pseudo-labels | SVM AUC: 0.0109 | RF AUC: 0.3269
Raw features   | Ground truth  | SVM AUC: 0.9985 | RF AUC: 1.0000
PCA-reduced    | Pseudo-labels | SVM AUC: 0.2270 | RF AUC: 0.4592
PCA-reduced    | Ground truth  | SVM AUC: 0.9916 | RF AUC: 0.9824


In [ ]:
#ClaudeCode Ver. 1: ROC curves (not just scalar AUC) for both raw and PCA-reduced classifiers,
#comparing pseudo-label-trained vs ground-truth-trained models side by side
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for probs, label in [
    (svm_probs_pseudo, f'SVM (pseudo) AUC={svm_auc_pseudo:.3f}'),
    (svm_probs_truth, f'SVM (truth) AUC={svm_auc_truth:.3f}'),
    (rf_probs_pseudo, f'RF (pseudo) AUC={rf_auc_pseudo:.3f}'),
    (rf_probs_truth, f'RF (truth) AUC={rf_auc_truth:.3f}'),
]:
    fpr, tpr, _ = roc_curve(y2_holdout, probs)
    axes[0].plot(fpr, tpr, label=label)
axes[0].plot([0, 1], [0, 1], linestyle='--', color='gray', label='Chance')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves: Raw Features (371k probes)')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

for probs, label in [
    (svm_probs_pseudo_pca, f'SVM (pseudo) AUC={svm_auc_pseudo_pca:.3f}'),
    (svm_probs_truth_pca, f'SVM (truth) AUC={svm_auc_truth_pca:.3f}'),
    (rf_probs_pseudo_pca, f'RF (pseudo) AUC={rf_auc_pseudo_pca:.3f}'),
    (rf_probs_truth_pca, f'RF (truth) AUC={rf_auc_truth_pca:.3f}'),
]:
    fpr, tpr, _ = roc_curve(y2_holdout, probs)
    axes[1].plot(fpr, tpr, label=label)
axes[1].plot([0, 1], [0, 1], linestyle='--', color='gray', label='Chance')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title(f'ROC Curves: PCA-Reduced Features (n={pca_model.n_components_})')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/processed/roc_curves_raw_vs_pca.png', dpi=150)
plt.show()